In [198]:
from lark import Lark, ast_utils, Transformer, v_args

from dataclasses import dataclass

from lark.tree import Meta

import sys

In [199]:
this_module = sys.modules[__name__]

print(this_module)

<module '__main__'>


In [200]:
# source_code = '''
# let x = 5
# let y = 10

# def add(a, b):
#     return a + b
# end

# def func():
#     # TODO: Implement this function
# end


# let result = add(x, y)
# '''

In [212]:
source_code = """
let x = 5

"""

In [213]:
grammar = r'''
    start: code_block

    code_block: statement+

    statement: let_statement
             | function_definition
             | return_statement
             | print_statement

    let_statement: "let" NAME "=" expr

    function_definition: "def" NAME "(" [args] ")" ":" statement* "end"

    print_statement: "print" "(" expr ")"


    args: expr ("," expr)*

    call: NAME "(" [args] ")"

    expr: NAME
        | NUMBER
        | expr "+" expr
        | call

    return_statement: "return" expr
    COMMENT: /#.*/
    NEWLINE: /(\r?\n)+/
    %ignore COMMENT
    %ignore NEWLINE
    %import common.CNAME -> NAME
    %import common.NUMBER
    %import common.WS
    %ignore WS
'''

In [214]:
def get_parser(grammar : str, parser_type: str = 'lalr', maybe_placeholders: bool = True):
    # Create and return a Lark parser instance using the specified grammar
    # Initialize the parser with the grammar and LALR parser algorithm
    return Lark(grammar=grammar, parser=parser_type, maybe_placeholders=maybe_placeholders)

In [215]:
# Test the parser with the source code
parsed_tree = get_parser(grammar).parse(source_code)
print(parsed_tree.pretty())

start
  code_block
    statement
      let_statement
        x
        expr	5



In [221]:
# Transform the parse tree into a more usable format (Abstract Syntax Tree)


class _AST(ast_utils.Ast):
    # Base class for all AST nodes
    pass 

class _Statement(_AST):
    # Base class for all statement nodes
    pass

class _Expression(_AST):
    # Base class for all expression nodes
    pass

@dataclass
class Name:
    name: str

@dataclass 
class NUMBER(_Expression, ast_utils.WithMeta):
    value: int
    meta: Meta

# class Expression(_AST, ast_utils.WithMeta):
#     meta: Meta
#     value: any

@dataclass
class PrintStatement(_Statement):
    expr: _Expression


@dataclass
class LetVariableStatement(_Statement):
    name: str
    value: _Expression

@dataclass
class FunctionDefinitionStatement(_Statement):
    name: str
    parameters: list[str]
    body: list[_Statement]

@dataclass
class ReturnStatement(_Statement):
    value: _Expression

@dataclass
class CallExpression(_Expression):
    name: str
    arguments: list[_Expression]

@dataclass 
class CodeBlock(_AST, ast_utils.AsList):
    statements: list[_Statement]



class ToAst(Transformer):
    def code_block(self, items):
        return CodeBlock(statements=items)

    def let_statement(self, items):
        name = items[0]
        value = items[1]
        return LetVariableStatement(name=name, value=value)

    def function_definition(self, items):
        name = items[0]
        parameters = items[1] if len(items) > 2 else []
        body = items[-1]
        return FunctionDefinitionStatement(name=name, parameters=parameters, body=body)

    def return_statement(self, items):
        return ReturnStatement(value=items[0])

    def print_statement(self, items):
        return PrintStatement(expr=items[0])

    def call(self, items):
        name = items[0]
        arguments = items[1] if len(items) > 1 else []
        return CallExpression(name=name, arguments=arguments)

    def expr(self, items):
        if len(items) == 1:
            return items[0]
        elif len(items) == 3:
            left = items[0]
            op = items[1]
            right = items[2]
            if op == '+':
                return ast_utils.BinOp(left=left, op=op, right=right)
        return items[0]

    @v_args(inline=True)
    def start(self, x):
        return x

In [222]:
transformer = ast_utils.create_transformer(this_module, ToAst())

In [223]:
def parse(source_code, parser: Lark):
    tree = parser.parse(source_code)
    return transformer.transform(tree)

In [224]:
parser = get_parser(grammar)
ast = parse(source_code, parser)
ast

CodeBlock(statements=[Tree(Token('RULE', 'statement'), [LetVariableStatement(name=Token('NAME', 'x'), value=Token('NUMBER', '5'))])])

CodeBlock(statements=[SetVar(name=Token('NAME', 'a'), value=Value(meta=<lark.tree.Meta object at 0x000001C8C8D9DC70>, value=1)), If(cond=Value(meta=<lark.tree.Meta object at 0x000001C8C9382C10>, value=Name(name=Token('NAME', 'a'))), then=CodeBlock(statements=[Print(value=Value(meta=<lark.tree.Meta object at 0x000001C8C9383CA0>, value='a is 1')), SetVar(name=Token('NAME', 'a'), value=Value(meta=<lark.tree.Meta object at 0x000001C8C9383750>, value=2))]))])
